# depthfly — depth 카메라(Gemini 2L) 하나 + 초파리 커넥톰만

RGB·LiDAR·CNN·MLP 없음. **depth 가까움 영상 3프레임(30 Hz)** → 시각엽(T4/T5 → HS/VS·LPLC2·LC·L3) → 하행 뉴런 48 → 선형 읽기 → 조향·속도.

**v3 (이 노트북):** 회로 극성 수정 (다가옴 LPLC2 가 2 m 밖 물체에서 거꾸로 나오던 것, DN 부호 랜덤 → 좌우 대칭 규칙) + **레이싱라인 모방학습으로 시작 → SAC**.
v2 모델과 호환 안 됨 → 저장 폴더 `depthfly_v3b` (v3 첫 실행은 행동 잡음이 커서 중단).

**순서:** ① → ② → ③ → ④ (12/12, 극성 11/11) → ⑤ → ⑤-2 모방학습 → ⑥ 학습.
**세션이 끊기면:** ①②③ 과 ⑥ 만 다시 (모방학습 결과·`last_model.zip` 은 드라이브에 있어서 이어 감).


## ① GPU 확인

In [ ]:
!nvidia-smi -L || echo "GPU 없음 → 런타임 유형을 T4 GPU 로 바꾸세요"
!python --version

## ② 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/mapless40_runs/depthfly_v3b'
import os; os.makedirs(RUN_DIR, exist_ok=True); print(RUN_DIR)

V_MAX = 8.0        # 최고 속도 [m/s] (최저 2). 모방학습·학습·평가·GIF 가 같은 값을 써야 함
BRAKE = ''         # 능동 브레이크로 학습하려면 '--brake' (실차는 지금 타력 감속만)


## ③ 코드 받기 + 패키지 설치

In [ ]:
%cd /content
!test -d 2027_F1tenth_test || git clone -q --depth 1 -b feat/mapless40-asym-sac https://github.com/tkddn647-ship-it/2027_F1tenth_test
%cd /content/2027_F1tenth_test
!git pull -q origin feat/mapless40-asym-sac
!git log --oneline -1
!pip -q install "stable-baselines3>=2.3" "gymnasium>=0.29" scipy pyyaml pillow tensorboard

## ④ 테스트 (numpy 9개 + torch 3개 → `12/12 passed`) + 회로 극성 검사 (`11/11 OK`)

In [ ]:
!python -m camera.depthfly.tests
!python -m camera.depthfly.polarity


## ⑤ depth 파리 눈이 보는 것 (학습 전, pure pursuit 로 주행)

In [ ]:
!python -m camera.depthfly.viz --map ifac --seconds 6 --obstacles 1 --out /content/depthfly_eye.gif
from IPython.display import Image, display
display(Image('/content/depthfly_eye.gif'))


## ⑤-2 레이싱라인 모방학습 (DAgger)

선생님 = 레이싱라인 pure pursuit (맵을 아는 운전자). 학생 = 실차에 올라갈 회로 그대로. 학생이 직접 몰다 벗어난 상황에도 '선생님이라면' 정답을 붙여 배움.

- `USE_PRETRAINED = True` (기본): 레포에 있는 결과 `camera/depthfly/results/bc_init_v3.npz` 를 복사 (바로 끝남, V_MAX 8 기준).
- `False`: 여기서 새로 모방학습 (CPU 약 18분). V_MAX 를 바꿨으면 반드시 False.

마지막 줄에 출발 위치 8곳 중 몇 곳이 3바퀴 완주했는지 나온다 (레포 결과: 5/8, 최고랩 ifac 12.8 s · 팀 맵 11.5 s, 장애물 없이).

In [ ]:
import os, shutil
USE_PRETRAINED = True
BC_NPZ = f'{RUN_DIR}/bc_init.npz'
if not os.path.exists(BC_NPZ):
    if USE_PRETRAINED and V_MAX == 8.0 and not BRAKE:
        shutil.copy('camera/depthfly/results/bc_init_v3.npz', BC_NPZ); print('레포의 모방학습 결과 사용')
    else:
        !python -m camera.depthfly.bc --out "{BC_NPZ}" --v-max {V_MAX} {BRAKE}
!python -m camera.depthfly.bc_eval --npz "{BC_NPZ}"


## ⑥ SAC 학습 (모방학습 결과에서 시작)

처음 2만 스텝은 모방한 정책으로 운전하며 데이터를 모으고, 이어 3만 스텝은 **critic 만** 학습 (actor 고정) → 그 뒤 전체 학습.
150분 뒤 스스로 저장하고 멈춘다. 평가는 5만 스텝마다 (`best_model.zip`, `best.json`). 다시 실행하면 이어서 학습.

In [ ]:
MAPS = 'ifac:3,roboracer_0817:3,Spielberg:1,Silverstone:1,Monza:1'
EVAL_MAPS = 'ifac,roboracer_0817'
STEPS_PER_SESSION = 1_000_000
TIME_LIMIT_MIN = 150
N_ENVS = 4
BC = f'--bc-init "{RUN_DIR}/bc_init.npz"' if os.path.exists(f'{RUN_DIR}/bc_init.npz') else ''   # 모방학습 결과로 시작
print('시작점:', BC or '무작위 (모방학습 없음)')

!python -m camera.depthfly.train --maps {MAPS} --eval-maps {EVAL_MAPS} \
   --timesteps {STEPS_PER_SESSION} --n-envs {N_ENVS} --subproc --device cuda \
   --time-limit-min {TIME_LIMIT_MIN} --save-dir "{RUN_DIR}" --resume auto \
   --v-max {V_MAX} {BRAKE} {BC}


## ⑥-2 진행 확인 (학습 셀이 끝난 뒤, 또는 다른 세션에서)

In [ ]:
import json, os
for f in ('best.json',):
    q = os.path.join(RUN_DIR, f)
    print(f, '→', json.load(open(q)) if os.path.exists(q) else '아직 없음')
print(sorted(os.listdir(RUN_DIR)))


## ⑦ 평가 (장애물 없이 / 2개)

In [ ]:
!python -m camera.depthfly.evaluate --model "{RUN_DIR}/best_model.zip" --maps ifac,roboracer_0817 --spawns 4 --laps 3 --obstacles 0 --v-max {V_MAX} {BRAKE}
!python -m camera.depthfly.evaluate --model "{RUN_DIR}/best_model.zip" --maps ifac,roboracer_0817 --spawns 4 --laps 3 --obstacles 2 --v-max {V_MAX} {BRAKE}


## ⑧ 학습된 정책 주행 + 회로 활동 GIF

In [ ]:
!python -m camera.depthfly.viz --map ifac --model "{RUN_DIR}/best_model.zip" --seconds 12 --obstacles 1 --out /content/depthfly_trained.gif --v-max {V_MAX} {BRAKE}
from IPython.display import Image, display
display(Image('/content/depthfly_trained.gif'))
